In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

# set display options to see more data
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Path to the file 
file_path = '/Users/abyudhka/Desktop/Project/australian-labor-market-analysis/Data/Raw/MRM1.xlsx'

# See file structure
excel_file = pd.ExcelFile(file_path) # to look at file 
print("Available sheets:")
for i, sheet in enumerate(excel_file.sheet_names, 1): # create and print number and sheet pairs
    print(f" {i}. {sheet}")

Available sheets:
 1. Contents
 2. Table 1
 3. Table 2
 4. Table 3
 5. Table 4
 6. Table 5
 7. Table 6


In [3]:
# Load Contents sheet
df_contents = pd.read_excel(file_path, sheet_name = 'Contents')

# Display first 20 rows
print("Contents sheet structure:")
print(df_contents.head(20))

Contents sheet structure:
                      Australian Bureau of Statistics  \
0   6291.0.55.001 - Modelled estimates of labour f...   
1   Released at 11.30 am (Canberra time) 29 Januar...   
2                                                 NaN   
3                                                 NaN   
4                                                 NaN   
5                                                 NaN   
6                                                 NaN   
7                                                 NaN   
8                                                 NaN   
9                                                 NaN   
10                                                NaN   
11                                                NaN   
12                                                NaN   
13                                                NaN   
14                                                NaN   
15                                                NaN   
16   

In [4]:
# Load first 15 rows of table 1 with no header for raw structure
df_raw = pd.read_excel(file_path, sheet_name = 'Table 1', nrows= 15, header = None) # To display data as is in file

print("Table 1 - Raw structure:")
print(df_raw)

print("\nRow 0 (first row):")
print(df_raw.iloc[0])
print(df_raw.shape)

Table 1 - Raw structure:
                                                  0                    1    \
0                     Australian Bureau of Statistics                  NaN   
1   6291.0.55.001 - Modelled estimates of labour f...                  NaN   
2   Released at 11.30 am (Canberra time) 29 Januar...                  NaN   
3   Table 1 - Employed persons ('000) by SA4, Janu...                  NaN   
4                                                 SA4  2025-12-01 00:00:00   
5                                   102 Central Coast           173.006524   
6          115 Sydney - Baulkham Hills and Hawkesbury           161.997793   
7                              116 Sydney - Blacktown           246.956782   
8                   117 Sydney - City and Inner South           232.696019   
9                        118 Sydney - Eastern Suburbs           170.660487   
10                      119 Sydney - Inner South West           341.151935   
11                            120 Sydne

In [4]:
## Identify all the major cities SA4 regions
df_cities = pd.read_excel(file_path, sheet_name='Table 1', skiprows=4)



# Look for major cities regions
major_cities = df_cities[
    df_cities['SA4'].str.contains('Sydney|Melbourne|Brisbane|Adelaide|Perth', case = False, na=False)
]

# Check result
print(f"\n Found{len(major_cities)} major cities:")
print(major_cities['SA4'].tolist())
print(major_cities.shape)


 Found36 major cities:
['115 Sydney - Baulkham Hills and Hawkesbury', '116 Sydney - Blacktown', '117 Sydney - City and Inner South', '118 Sydney - Eastern Suburbs', '119 Sydney - Inner South West', '120 Sydney - Inner West', '121 Sydney - North Sydney and Hornsby', '122 Sydney - Northern Beaches', '123 Sydney - Outer South West', '124 Sydney - Outer West and Blue Mountains', '125 Sydney - Parramatta', '126 Sydney - Ryde', '127 Sydney - South West', '128 Sydney - Sutherland', '206 Melbourne - Inner', '207 Melbourne - Inner East', '208 Melbourne - Inner South', '209 Melbourne - North East', '210 Melbourne - North West', '211 Melbourne - Outer East', '212 Melbourne - South East', '213 Melbourne - West', '301 Brisbane - East', '302 Brisbane - North', '303 Brisbane - South', '304 Brisbane - West', '305 Brisbane Inner City', '401 Adelaide - Central and Hills', '402 Adelaide - North', '403 Adelaide - South', '404 Adelaide - West', '503 Perth - Inner', '504 Perth - North East', '505 Perth - N

In [5]:

#defining codes for each major city
major_cities_sa4 = {
    'Sydney': list(range(115,129)),
    'Melbourne': list(range(206,214)),
    'Brisbane': list(range(301,306)),
    'Adelaide': list(range(401,405)),
    'Perth': list(range(503,508))
}
                           

# Flatten to single list
all_major_city_sa4_codes = []
for city, codes in major_cities_sa4.items():
    all_major_city_sa4_codes.extend(codes)

print("Major city SA4 codes defined:")
print(f"Total SA4 regions: {len(all_major_city_sa4_codes)}")
print(f"Sydney: {len(major_cities_sa4['Sydney'])} SA4s")
print(f"Melbourne: {len(major_cities_sa4['Melbourne'])} SA4s")
print(f"Brisbane: {len(major_cities_sa4['Brisbane'])} SA4s")
print(f"Adelaide: {len(major_cities_sa4['Adelaide'])} SA4s")
print(f"Perth: {len(major_cities_sa4['Perth'])} SA4s")

Major city SA4 codes defined:
Total SA4 regions: 36
Sydney: 14 SA4s
Melbourne: 8 SA4s
Brisbane: 5 SA4s
Adelaide: 4 SA4s
Perth: 5 SA4s


In [21]:
df_emp = pd.read_excel(file_path, sheet_name = 'Table 1', skiprows = 4)
# extracting the SA4 code number from SA4 column
df_emp['sa4_code'] = pd.to_numeric(
    df_emp['SA4'].str.extract(r'^(\d+)')[0],  # extracting text matching a regex pattern
    errors='coerce'
).astype('Int64')

# filter for all major cities using the SA4 codes defined
df_emp_major_cities = df_emp[
     df_emp['sa4_code'].isin(all_major_city_sa4_codes)
].copy()

print(f"Filtered to {len(df_emp_major_cities)} major city SA4 regions")

#Basic melt with custom column names
df_emp_long = df_emp_major_cities.melt(
    id_vars=['sa4_code', 'SA4'],  ## why square brackets
    var_name='date',
    value_name ='employed_persons'
)

## Clean after melting
# rename SA4 column
df_emp_long = df_emp_long.rename(columns={'SA4': 'sa4_name'})
# convert date from text to datetime
df_emp_long['date'] = pd.to_datetime(df_emp_long['date'])

# filter for 2019 onwards
df_emp_long = df_emp_long[df_emp_long['date'] >= '2019-01-01'].copy() ## why copy

# convert values from thousands to actual numbers
df_emp_long['employed_persons'] = pd.to_numeric(df_emp_long['employed_persons'], errors='coerce')
df_emp_long['employed_persons'] = df_emp_long['employed_persons']*1000
df_emp_long['employed_persons'] = df_emp_long['employed_persons'].round(0).astype('Int64')

# add city column
def get_city_from_sa4_code(sa4_code):
    """Map SA4 code to city name"""
    if 101 <= sa4_code <= 128:
        return 'Sydney'
    elif 206 <= sa4_code <= 213:
        return 'Melbourne'
    elif 301 <= sa4_code <= 305:
        return 'Brisbane'
    elif 401 <= sa4_code <= 404:
        return 'Adelaide'
    elif 503 <= sa4_code <= 507:
        return 'Perth'
    else:
        return 'Other'

df_emp_long['city'] = df_emp_long['sa4_code'].apply(get_city_from_sa4_code)

# reorder columns
df_emp_long = df_emp_long[['city', 'sa4_code', 'sa4_name', 'date', 'employed_persons']]

#sort table
df_emp_long = df_emp_long.sort_values(['city', 'sa4_code', 'date']).reset_index(drop=True)

df_employment =df_emp_long.copy()

print("\nEmployement data cleaned:")
print(df_employment.head(10))
print("\nCity distribution:")
print(df_employment['city'].value_counts())


Filtered to 36 major city SA4 regions

Employement data cleaned:
       city  sa4_code                          sa4_name       date  \
0  Adelaide       401  401 Adelaide - Central and Hills 2019-01-01   
1  Adelaide       401  401 Adelaide - Central and Hills 2019-02-01   
2  Adelaide       401  401 Adelaide - Central and Hills 2019-03-01   
3  Adelaide       401  401 Adelaide - Central and Hills 2019-04-01   
4  Adelaide       401  401 Adelaide - Central and Hills 2019-05-01   
5  Adelaide       401  401 Adelaide - Central and Hills 2019-06-01   
6  Adelaide       401  401 Adelaide - Central and Hills 2019-07-01   
7  Adelaide       401  401 Adelaide - Central and Hills 2019-08-01   
8  Adelaide       401  401 Adelaide - Central and Hills 2019-09-01   
9  Adelaide       401  401 Adelaide - Central and Hills 2019-10-01   

   employed_persons  
0            155469  
1            159362  
2            161148  
3            161895  
4            162471  
5            162062  
6         

In [22]:
## Clean table 2
df_unemp = pd.read_excel(file_path, sheet_name = 'Table 2', skiprows = 4)

# extracting the sa4 codes
df_unemp['sa4_code'] = pd.to_numeric(
    df_unemp['SA4'].str.extract(r'^(\d+)')[0],
    errors='coerce'
).astype('Int64')

# filter for all major cities 
df_unemp_major_cities = df_unemp[
   df_unemp['sa4_code'].isin(all_major_city_sa4_codes)
].copy()

print(f"Filtered to {len(df_unemp_major_cities)} major city SA4 regions")

# Melt function for table 2
df_unemp_long = df_unemp_major_cities.melt(
    id_vars = ['sa4_code', 'SA4'],
    var_name = 'date', 
    value_name = 'unemployed_persons'
)

# clean after melting
#rename sa4 column
df_unemp_long = df_unemp_long.rename(columns={'SA4':'sa4_name'})
# convert date from text to datetime
df_unemp_long['date'] = pd.to_datetime(df_unemp_long['date'])

# filter for 2019 onwards
df_unemp_long = df_unemp_long[df_unemp_long['date'] >= '2019-01-01'].copy()

# convert values from thousands to actual numbers
df_unemp_long['unemployed_persons'] = pd.to_numeric(df_unemp_long['unemployed_persons'], errors='coerce')
df_unemp_long['unemployed_persons'] = df_unemp_long['unemployed_persons']*1000
df_unemp_long['unemployed_persons'] = df_unemp_long['unemployed_persons'].round(0).astype('Int64')

# add the city column
def get_city_from_sa4_code(sa4_code):
    """Map SA4 code to city name"""
    if 101 <= sa4_code <= 128:
        return 'Sydney'
    elif 206 <= sa4_code <= 213:
        return 'Melbourne'
    elif 301 <= sa4_code <= 305:
        return 'Brisbane'
    elif 401 <= sa4_code <= 404:
        return 'Adelaide'
    elif 503 <= sa4_code <= 507:
        return 'Perth'
    else:
        return 'Other'

df_unemp_long['city'] = df_unemp_long['sa4_code'].apply(get_city_from_sa4_code)

# reorder columns
df_unemp_long = df_unemp_long[['city', 'sa4_code', 'sa4_name', 'date', 'unemployed_persons']]

# sort table
df_unemp_long = df_unemp_long.sort_values(['city', 'sa4_code', 'date']).reset_index(drop=True)

# storing data for merging
df_unemployment = df_unemp_long.copy()
print(df_unemp_long)

print("\nUnemployment data cleaned:")
print(df_unemployment.head(10))
print("\nCity distribution:")
print(df_unemployment['city'].value_counts())



Filtered to 36 major city SA4 regions
          city  sa4_code                          sa4_name       date  \
0     Adelaide       401  401 Adelaide - Central and Hills 2019-01-01   
1     Adelaide       401  401 Adelaide - Central and Hills 2019-02-01   
2     Adelaide       401  401 Adelaide - Central and Hills 2019-03-01   
3     Adelaide       401  401 Adelaide - Central and Hills 2019-04-01   
4     Adelaide       401  401 Adelaide - Central and Hills 2019-05-01   
...        ...       ...                               ...        ...   
3019    Sydney       128           128 Sydney - Sutherland 2025-08-01   
3020    Sydney       128           128 Sydney - Sutherland 2025-09-01   
3021    Sydney       128           128 Sydney - Sutherland 2025-10-01   
3022    Sydney       128           128 Sydney - Sutherland 2025-11-01   
3023    Sydney       128           128 Sydney - Sutherland 2025-12-01   

      unemployed_persons  
0                   7949  
1                   7386  
2   

In [32]:
## Clean table 5
# drop metadeta
df_rate = pd.read_excel(file_path, sheet_name = 'Table 5', skiprows = 4)

# Extract SA4 code number from SA4 column
df_rate['sa4_code'] = pd.to_numeric(
    df_rate['SA4'].str.extract(r'^(\d+)')[0],
    errors='coerce'
).astype('Int64')

# filter for all major citites  
df_rate_major_cities = df_rate[
     df_rate['sa4_code'].isin(all_major_city_sa4_codes)
].copy()

print(f"Filtered to {len(df_rate_major_cities)} major city SA4 regions")

# melt table 5
df_rate_long =df_rate_major_cities.melt(
    id_vars = ['sa4_code', 'SA4'],
    var_name = 'date',
    value_name = 'unemployment_rate'
)

# clean after melting 
# rename sa4 column
df_rate_long = df_rate_long.rename(columns={'SA4':'sa4_name'})

#convert date from text to datetime
df_rate_long['date'] = pd.to_datetime(df_rate_long['date'])

# filter for 2019
df_rate_long = df_rate_long[df_rate_long['date'] >= '2019-01-01'].copy()

# clean values
df_rate_long['unemployment_rate'] = pd.to_numeric(df_rate_long['unemployment_rate'], errors = 'coerce') ## reduce decimal places

# Add city column
def get_city_from_sa4_code(sa4_code):
    """Map SA4 code to city name"""
    if 101 <= sa4_code <= 127:
        return 'Sydney'
    elif 201 <= sa4_code <= 221:
        return 'Melbourne'
    elif 301 <= sa4_code <= 318:
        return 'Brisbane'
    elif 401 <= sa4_code <= 404:
        return 'Adelaide'
    elif 501 <= sa4_code <= 512:
        return 'Perth'
    else:
        return 'Other'

df_rate_long['city'] = df_rate_long['sa4_code'].apply(get_city_from_sa4_code)

# reorder columns
df_rate_long = df_rate_long[['city', 'sa4_code', 'sa4_name', 'date', 'unemployment_rate']]

# sort table 
df_rate_long = df_rate_long.sort_values(['city', 'sa4_code', 'date']).reset_index(drop=True)

# store data for merging
df_unemployment_rate = df_rate_long.copy()

print("\nUnemployment rate data cleaned:")
print(df_unemployment_rate.head(10))
print("\nCity distribution:")
print(df_unemployment_rate['city'].value_counts())




Filtered to 36 major city SA4 regions

Unemployment rate data cleaned:
       city  sa4_code                          sa4_name       date  \
0  Adelaide       401  401 Adelaide - Central and Hills 2019-01-01   
1  Adelaide       401  401 Adelaide - Central and Hills 2019-02-01   
2  Adelaide       401  401 Adelaide - Central and Hills 2019-03-01   
3  Adelaide       401  401 Adelaide - Central and Hills 2019-04-01   
4  Adelaide       401  401 Adelaide - Central and Hills 2019-05-01   
5  Adelaide       401  401 Adelaide - Central and Hills 2019-06-01   
6  Adelaide       401  401 Adelaide - Central and Hills 2019-07-01   
7  Adelaide       401  401 Adelaide - Central and Hills 2019-08-01   
8  Adelaide       401  401 Adelaide - Central and Hills 2019-09-01   
9  Adelaide       401  401 Adelaide - Central and Hills 2019-10-01   

   unemployment_rate  
0           4.864031  
1           4.429533  
2           4.350153  
3           4.589305  
4           4.061827  
5           4.150907

In [33]:
## Clean table 6
# drop metadeta
df_part = pd.read_excel(file_path, sheet_name = 'Table 6', skiprows = 4)

# Extract SA4 code number from SA4 column
df_part['sa4_code'] = pd.to_numeric(
    df_part['SA4'].str.extract(r'^(\d+)')[0],
    errors='coerce'
).astype('Int64')

# filter for adelaide regions 
df_part_major_cities = df_part[
     df_part['sa4_code'].isin(all_major_city_sa4_codes)
].copy()

print(f"Filtered to {len(df_part_major_cities)} major city SA4 regions")

# melt table 6
df_part_long =df_part_major_cities.melt(
    id_vars = ['sa4_code', 'SA4'],
    var_name = 'date',
    value_name = 'participation_rate'
)

# clean after melting 
# rename sa4 column
df_part_long = df_part_long.rename(columns={'SA4':'sa4_name'})
#convert date from text to datetime
df_part_long['date'] = pd.to_datetime(df_part_long['date'])

# filter for 2019
df_part_long = df_part_long[df_part_long['date'] >= '2019-01-01'].copy()

# clean values
df_part_long['participation_rate'] = pd.to_numeric(df_part_long['participation_rate'], errors = 'coerce') ## reduce decimal places

# Add city column
def get_city_from_sa4_code(sa4_code):
    """Map SA4 code to city name"""
    if 101 <= sa4_code <= 127:
        return 'Sydney'
    elif 201 <= sa4_code <= 221:
        return 'Melbourne'
    elif 301 <= sa4_code <= 318:
        return 'Brisbane'
    elif 401 <= sa4_code <= 404:
        return 'Adelaide'
    elif 501 <= sa4_code <= 512:
        return 'Perth'
    else:
        return 'Other'

df_part_long['city'] = df_part_long['sa4_code'].apply(get_city_from_sa4_code)

# reorder columns
df_part_long = df_part_long[['city', 'sa4_code', 'sa4_name', 'date', 'participation_rate']]

# sort table 
df_part_long = df_part_long.sort_values(['city', 'sa4_code', 'date']).reset_index(drop=True)

# store for merging
df_participation_rate = df_part_long.copy()

print("\nParticipation_rate:")
print(df_participation_rate.head(10))
print("\nCity distribution:")
print(df_participation_rate['city'].value_counts())

Filtered to 36 major city SA4 regions

Participation_rate:
       city  sa4_code                          sa4_name       date  \
0  Adelaide       401  401 Adelaide - Central and Hills 2019-01-01   
1  Adelaide       401  401 Adelaide - Central and Hills 2019-02-01   
2  Adelaide       401  401 Adelaide - Central and Hills 2019-03-01   
3  Adelaide       401  401 Adelaide - Central and Hills 2019-04-01   
4  Adelaide       401  401 Adelaide - Central and Hills 2019-05-01   
5  Adelaide       401  401 Adelaide - Central and Hills 2019-06-01   
6  Adelaide       401  401 Adelaide - Central and Hills 2019-07-01   
7  Adelaide       401  401 Adelaide - Central and Hills 2019-08-01   
8  Adelaide       401  401 Adelaide - Central and Hills 2019-09-01   
9  Adelaide       401  401 Adelaide - Central and Hills 2019-10-01   

   participation_rate  
0           63.647572  
1           64.832305  
2           65.390584  
3           65.784331  
4           65.583064  
5           65.405281  
6 

In [34]:
## Check all four tables

# Check structure of each dataframe
print("Employement shape:", df_employment.shape)
print("Unemployment shape:", df_unemployment.shape)
print("Unemployment rate shape:", df_unemployment_rate.shape)
print("Participation rate shape:", df_participation_rate.shape)

# Check date ranges match
print("\nDate ranges:")
print("Employment:", df_employment['date'].min(), "to", df_employment['date'].max())
print("Unemployment:", df_unemployment['date'].min(), "to", df_unemployment['date'].max())
# Should all be identical

# Check regions match
print("\nRegions in each table:")
print("Employment:", sorted(df_employment['sa4_name'].unique()))
print("Unemployment:", sorted(df_unemployment['sa4_name'].unique()))

Employement shape: (3024, 5)
Unemployment shape: (3024, 5)
Unemployment rate shape: (3024, 5)
Participation rate shape: (3024, 5)

Date ranges:
Employment: 2019-01-01 00:00:00 to 2025-12-01 00:00:00
Unemployment: 2019-01-01 00:00:00 to 2025-12-01 00:00:00

Regions in each table:
Employment: ['115 Sydney - Baulkham Hills and Hawkesbury', '116 Sydney - Blacktown', '117 Sydney - City and Inner South', '118 Sydney - Eastern Suburbs', '119 Sydney - Inner South West', '120 Sydney - Inner West', '121 Sydney - North Sydney and Hornsby', '122 Sydney - Northern Beaches', '123 Sydney - Outer South West', '124 Sydney - Outer West and Blue Mountains', '125 Sydney - Parramatta', '126 Sydney - Ryde', '127 Sydney - South West', '128 Sydney - Sutherland', '206 Melbourne - Inner', '207 Melbourne - Inner East', '208 Melbourne - Inner South', '209 Melbourne - North East', '210 Melbourne - North West', '211 Melbourne - Outer East', '212 Melbourne - South East', '213 Melbourne - West', '301 Brisbane - East'

In [35]:
## Merge all four tables

#merge table 1 & 2
df_merged = df_employment.merge(
    df_unemployment.drop(columns=['city', 'sa4_name']), ## to keep unique columns and avoid duplicate column conflict
    on=['date', 'sa4_code'],
    how='inner'
)

# check result
print("After first merge:")
print("Shape:", df_merged.shape)       
print("\nFirst few rows:")
print(df_merged.head())

# merge table 5 to merged dataframe
df_merged = df_merged.merge(
    df_unemployment_rate.drop(columns=['city', 'sa4_name']), 
    on=['date', 'sa4_code'],
    how='inner'
)
# check result
print("\nAfter second merge:")
print("Shape:", df_merged.shape)       
print("\nFirst few rows:")
print(df_merged.head())

# Merge Table 6 (Participation Rate)
df_merged = df_merged.merge(
    df_participation_rate.drop(columns=['city', 'sa4_name']),
    on=['date', 'sa4_code'],  
    how='inner'
)
# check result
print("\nAfter last merge:")
print("Shape:", df_merged.shape)
print(f"Columns: {df_merged.columns.tolist()}")
print("\nFirst few rows:")
print(df_merged.head())


After first merge:
Shape: (3024, 6)

First few rows:
       city  sa4_code                          sa4_name       date  \
0  Adelaide       401  401 Adelaide - Central and Hills 2019-01-01   
1  Adelaide       401  401 Adelaide - Central and Hills 2019-02-01   
2  Adelaide       401  401 Adelaide - Central and Hills 2019-03-01   
3  Adelaide       401  401 Adelaide - Central and Hills 2019-04-01   
4  Adelaide       401  401 Adelaide - Central and Hills 2019-05-01   

   employed_persons  unemployed_persons  
0            155469                7949  
1            159362                7386  
2            161148                7329  
3            161895                7787  
4            162471                6879  

After second merge:
Shape: (3024, 7)

First few rows:
       city  sa4_code                          sa4_name       date  \
0  Adelaide       401  401 Adelaide - Central and Hills 2019-01-01   
1  Adelaide       401  401 Adelaide - Central and Hills 2019-02-01   
2  Adelai

In [36]:
## Qualti checks
df_major_cities_final = df_merged.copy()

# Check for missing values (NaN)
print("\nMissing values check:")
print(df_major_cities_final.isnull().sum())

# Verify all 4 Adelaide regions are in final data
print("\nRegions check:")
print(df_major_cities_final['sa4_name'].unique())

# Verify date range
print("\nDate range check:")
print(f"Start: {df_major_cities_final['date'].min()}")
print(f"End: {df_major_cities_final['date'].max()}")
print(f"Total months: {df_major_cities_final['date'].nunique()}")



Missing values check:
city                  0
sa4_code              0
sa4_name              0
date                  0
employed_persons      0
unemployed_persons    0
unemployment_rate     0
participation_rate    0
dtype: int64

Regions check:
['401 Adelaide - Central and Hills' '402 Adelaide - North'
 '403 Adelaide - South' '404 Adelaide - West' '301 Brisbane - East'
 '302 Brisbane - North' '303 Brisbane - South' '304 Brisbane - West'
 '305 Brisbane Inner City' '206 Melbourne - Inner'
 '207 Melbourne - Inner East' '208 Melbourne - Inner South'
 '209 Melbourne - North East' '210 Melbourne - North West'
 '211 Melbourne - Outer East' '212 Melbourne - South East'
 '213 Melbourne - West' '503 Perth - Inner' '504 Perth - North East'
 '505 Perth - North West' '506 Perth - South East'
 '507 Perth - South West' '115 Sydney - Baulkham Hills and Hawkesbury'
 '116 Sydney - Blacktown' '117 Sydney - City and Inner South'
 '118 Sydney - Eastern Suburbs' '119 Sydney - Inner South West'
 '120 Sydney -

In [37]:
## Final clean and export
# Sort 
df_major_cities_final = df_major_cities_final.sort_values(
    ['city', 'sa4_code', 'date']
).reset_index(drop=True)

print("Final sorted data:")
print(df_major_cities_final.head(10))

# Save to processed folder
output_path = '/Users/abyudhka/Desktop/Project/australian-labor-market-analysis/Data/processed/employment_sa4_major_cities.csv'

df_major_cities_final.to_csv(output_path, index=False)
print(f" Saved to: {output_path}")
print(f"   Rows: {len(df_major_cities_final)}")
print(f"   Columns: {len(df_major_cities_final.columns)}")

# Verification: Read back
df_test = pd.read_csv(output_path)
print(f"\n Verification: Successfully read back {len(df_test)} rows")
print("\nFile preview:")
print(df_test.head())

Final sorted data:
       city  sa4_code                          sa4_name       date  \
0  Adelaide       401  401 Adelaide - Central and Hills 2019-01-01   
1  Adelaide       401  401 Adelaide - Central and Hills 2019-02-01   
2  Adelaide       401  401 Adelaide - Central and Hills 2019-03-01   
3  Adelaide       401  401 Adelaide - Central and Hills 2019-04-01   
4  Adelaide       401  401 Adelaide - Central and Hills 2019-05-01   
5  Adelaide       401  401 Adelaide - Central and Hills 2019-06-01   
6  Adelaide       401  401 Adelaide - Central and Hills 2019-07-01   
7  Adelaide       401  401 Adelaide - Central and Hills 2019-08-01   
8  Adelaide       401  401 Adelaide - Central and Hills 2019-09-01   
9  Adelaide       401  401 Adelaide - Central and Hills 2019-10-01   

   employed_persons  unemployed_persons  unemployment_rate  participation_rate  
0            155469                7949           4.864031           63.647572  
1            159362                7386         